# Argon saturation-property analysis

This notebook loads `argon_saturation.tsv`, downloaded from the [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/fluid/), validates the table, assigns concise column names, plots the main saturation properties, and provides interpolation helpers for simulation inputs.

The source query covers **85–140 K** in **0.1 K** increments. A suffix of `liquid` or `vapor` identifies the saturated phase.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

## Load and validate the NIST table

In [ ]:
data_path = Path("argon_saturation.tsv")
if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {data_path.resolve()}. Start Jupyter from the repository root, "
        "or change data_path to the file's location."
    )

raw = pd.read_csv(data_path, sep="\t")
print(f"Loaded {raw.shape[0]} rows and {raw.shape[1]} columns from {data_path}")
raw.head()

In [ ]:
column_names = [
    "temperature_K",
    "pressure_bar",
    "density_liquid_mol_L",
    "volume_liquid_L_mol",
    "internal_energy_liquid_kJ_mol",
    "enthalpy_liquid_kJ_mol",
    "entropy_liquid_J_mol_K",
    "cv_liquid_J_mol_K",
    "cp_liquid_J_mol_K",
    "sound_speed_liquid_m_s",
    "joule_thomson_liquid_K_bar",
    "viscosity_liquid_uPa_s",
    "thermal_conductivity_liquid_W_m_K",
    "surface_tension_N_m",
    "density_vapor_mol_L",
    "volume_vapor_L_mol",
    "internal_energy_vapor_kJ_mol",
    "enthalpy_vapor_kJ_mol",
    "entropy_vapor_J_mol_K",
    "cv_vapor_J_mol_K",
    "cp_vapor_J_mol_K",
    "sound_speed_vapor_m_s",
    "joule_thomson_vapor_K_bar",
    "viscosity_vapor_uPa_s",
    "thermal_conductivity_vapor_W_m_K",
]

if raw.shape[1] != len(column_names):
    raise ValueError(f"Expected {len(column_names)} NIST columns, found {raw.shape[1]}")

df = raw.copy()
df.columns = column_names
df = df.apply(pd.to_numeric, errors="raise")

assert df["temperature_K"].is_monotonic_increasing
assert not df.isna().any().any()
print(f"Temperature range: {df.temperature_K.min():.1f}–{df.temperature_K.max():.1f} K")
print(f"Missing values: {df.isna().sum().sum()}")
df.head()

## Add useful derived quantities

NIST reports pressure in bar and molar density in mol/L for this query. The following columns convert those values to SI and compute the saturated enthalpy of vaporization.

In [ ]:
ARGON_MOLAR_MASS_KG_MOL = 39.948e-3

df["pressure_Pa"] = df["pressure_bar"] * 1e5
df["density_liquid_kg_m3"] = (
    df["density_liquid_mol_L"] * 1000 * ARGON_MOLAR_MASS_KG_MOL
)
df["density_vapor_kg_m3"] = (
    df["density_vapor_mol_L"] * 1000 * ARGON_MOLAR_MASS_KG_MOL
)
df["enthalpy_vaporization_kJ_mol"] = (
    df["enthalpy_vapor_kJ_mol"] - df["enthalpy_liquid_kJ_mol"]
)

df[[
    "temperature_K", "pressure_bar", "pressure_Pa",
    "density_liquid_kg_m3", "density_vapor_kg_m3",
    "surface_tension_N_m", "enthalpy_vaporization_kJ_mol",
]].head()

## Quick numerical summary

In [ ]:
summary_columns = [
    "temperature_K",
    "pressure_bar",
    "density_liquid_kg_m3",
    "density_vapor_kg_m3",
    "surface_tension_N_m",
    "enthalpy_vaporization_kJ_mol",
]
df[summary_columns].describe().T

## Saturation-property plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

axes[0, 0].plot(df.temperature_K, df.pressure_bar, color="tab:blue")
axes[0, 0].set(xlabel="Temperature (K)", ylabel="Saturation pressure (bar)")

axes[0, 1].semilogy(
    df.temperature_K, df.density_liquid_kg_m3, label="Liquid", color="tab:blue"
)
axes[0, 1].semilogy(
    df.temperature_K, df.density_vapor_kg_m3, label="Vapor", color="tab:orange"
)
axes[0, 1].set(xlabel="Temperature (K)", ylabel=r"Density (kg m$^{-3}$)")
axes[0, 1].legend()

axes[1, 0].plot(df.temperature_K, 1e3 * df.surface_tension_N_m, color="tab:green")
axes[1, 0].set(xlabel="Temperature (K)", ylabel=r"Surface tension (mN m$^{-1}$)")

axes[1, 1].plot(
    df.temperature_K, df.enthalpy_vaporization_kJ_mol, color="tab:red"
)
axes[1, 1].set(xlabel="Temperature (K)", ylabel=r"$\Delta H_{vap}$ (kJ mol$^{-1}$)")

fig.suptitle("NIST argon saturation properties", fontsize=15)
plt.show()

## Interpolate properties at a simulation temperature

The tabulated spacing is 0.1 K. This helper linearly interpolates every numeric property at any temperature inside the downloaded range.

In [ ]:
def properties_at_temperature(temperature_K: float, table: pd.DataFrame = df) -> pd.Series:
    """Linearly interpolate every property at a temperature in kelvin."""
    temperatures = table["temperature_K"].to_numpy()
    if not temperatures[0] <= temperature_K <= temperatures[-1]:
        raise ValueError(
            f"temperature_K must be between {temperatures[0]} and {temperatures[-1]} K"
        )

    values = {
        column: np.interp(temperature_K, temperatures, table[column].to_numpy())
        for column in table.columns
    }
    return pd.Series(values, name=f"properties_at_{temperature_K:g}_K")

T_simulation = 100.0
properties_at_temperature(T_simulation)[[
    "pressure_bar",
    "density_liquid_kg_m3",
    "density_vapor_kg_m3",
    "surface_tension_N_m",
    "enthalpy_vaporization_kJ_mol",
]]

## Export the cleaned table

Run this cell when you want a comma-separated version with the concise names and derived SI columns.

In [ ]:
clean_path = Path("argon_saturation_clean.csv")
df.to_csv(clean_path, index=False)
print(f"Saved {len(df)} rows to {clean_path.resolve()}")